## Implemented Native RAG (from scratch)

RAG Query Engine for PsyWiz

Complete RAG pipeline implementation:
- Retrieves relevant chunks from vector database
- Generates answers using OpenRouter API
- Provides proper citations with source metadata
- Interactive query interface

In [ ]:
import os
import json
import chromadb
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass
from sentence_transformers import SentenceTransformer

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

@dataclass
class RetrievalResult: # structure for retrieval results
    
    chunk_id: str
    content: str
    score: float
    metadata: Dict
    section: str
    document_id: str
    paper_title: str

@dataclass
class RAGResponse: #structure for RAG response
    answer: str
    sources: List[RetrievalResult]
    query: str
    confidence: float

class SimpleOpenRouterLLM: #no LangChain inheritance; used simpel wrapper
    
    def __init__(self, model_name: str = "google/gemini-2.0-flash-exp:free"):
        self.model_name = model_name
        
        api_key = os.getenv("OPENROUTER_API_KEY")
        if not api_key:
            raise ValueError("OPENROUTER_API_KEY not found in environment variables")
            
        self.client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=api_key,
        )
        
        print(f"✅ Connected to OpenRouter with model: {model_name}")
    
    def generate(self, prompt: str) -> str:
        
        try:
            completion = self.client.chat.completions.create(
                extra_headers={
                    "HTTP-Referer": "https://psywiz.local",
                    "X-Title": "PsyWiz RAG Engine",
                },
                model=self.model_name,
                messages=[
                    {
                        "role": "system",
                        "content": "You are a medical research assistant. Provide accurate, evidence-based answers using only the provided research context. Always cite specific findings and acknowledge limitations."
                    },
                    {
                        "role": "user", 
                        "content": prompt
                    }
                ],
                temperature=0.2,
                max_tokens=4500
            )
            
            return completion.choices[0].message.content
            
        except Exception as e:
            return f"Error calling OpenRouter API: {str(e)}"

class RAGQueryEngine:
    
    def __init__(self, 
                 vector_db_path: str = "D:/PsyWiz/vector_db",
                 collection_name: str = "psywiz_papers",
                 embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2",
                 retrieval_k: int = 6,
                 llm_model: str = "google/gemini-2.0-flash-exp:free"):
        
        self.vector_db_path = vector_db_path
        self.collection_name = collection_name
        self.retrieval_k = retrieval_k
        self.llm_model = llm_model
        
        print("🔧 Initializing RAG Query Engine with OpenRouter...")
        self._initialize_embeddings(embedding_model)
        self._initialize_vector_store()
        self._initialize_llm()
        
        print("✅ RAG Query Engine ready!")
    
    def _initialize_embeddings(self, model_name: str):
        
        print(f"📊 Loading embedding model: {model_name}")
        self.embedding_model = SentenceTransformer(model_name)
    
    def _initialize_vector_store(self):
        
        print(f"🗄️  Connecting to vector database: {self.vector_db_path}")
        
        # Connect to existing Chroma database
        client = chromadb.PersistentClient(path=self.vector_db_path)
        self.collection = client.get_collection(name=self.collection_name)
        
        print(f"📚 Found {self.collection.count()} documents in vector database")
    
    def _initialize_llm(self):
        
        self.llm = SimpleOpenRouterLLM(model_name=self.llm_model)
    
    def retrieve_relevant_chunks(self, query: str, k: int = None) -> List[RetrievalResult]:
       
        if k is None:
            k = self.retrieval_k
        
        # direct Chroma query (simpler approach)
        results = self.collection.query(
            query_texts=[query],
            n_results=k,
            include=['documents', 'metadatas', 'distances']
        )
        
        retrieval_results = []         # convert to RetrievalResult objects
        for doc, metadata, distance in zip(
            results['documents'][0],
            results['metadatas'][0], 
            results['distances'][0]
        ):
            result = RetrievalResult(
                chunk_id=metadata.get('chunk_id', 'unknown'),
                content=doc,
                score=1 - distance,  # convertig dist to similarity score
                metadata=metadata,
                section=metadata.get('section', 'Unknown'),
                document_id=metadata.get('document_id', 'unknown'),
                paper_title=metadata.get('paper_title', 'Unknown Paper')
            )
            retrieval_results.append(result)
        
        return retrieval_results
    
    def query(self, question: str, include_sources: bool = True) -> RAGResponse:
        print(f"Processing query: {question}")
        
        # Step 1: Retrieve relevant chunks
        retrieved_chunks = self.retrieve_relevant_chunks(question)
        # print(f"📖 Retrieved {len(retrieved_chunks)} relevant chunks")
        
        # Step 2: Build context from retrieved chunks
        context_parts = []
        for i, chunk in enumerate(retrieved_chunks, 1):
            context_parts.append(f"Source {i} ({chunk.document_id} - {chunk.section}):\n{chunk.content}")
        
        context = "\n\n".join(context_parts)
        
        # Step 3: Create prompt
        prompt = f"""You are an expert medical research assistant. Answer the question based ONLY on the provided research paper excerpts.

INSTRUCTIONS:
- Provide accurate, evidence-based answers
- Cite specific findings from the papers using source numbers (e.g., "Source 1", "Source 2")
- Include statistical data when available (prevalence rates, confidence intervals, etc.)
- Acknowledge limitations or gaps in the provided evidence
- If the context doesn't contain sufficient information, clearly state this
- Use a professional, academic tone

RESEARCH CONTEXT:
{context}

QUESTION: {question}

EVIDENCE-BASED ANSWER:"""
        
        # Step 4: Generate answer using LLM
        answer = self.llm.generate(prompt)
        
        # Step 5: Calculate confidence
        avg_score = sum(chunk.score for chunk in retrieved_chunks) / len(retrieved_chunks)
        confidence = min(avg_score * 1.2, 1.0)
        
        # Step 6: Create response
        response = RAGResponse(
            answer=answer,
            sources=retrieved_chunks if include_sources else [],
            query=question,
            confidence=confidence
        )
        
        return response
    
    def interactive_query(self):
        
        print("\n" + "="*60)
        print("🧠 PsyWiz RAG Query Engine - Interactive Mode")
        print("="*60)
        print("Ask questions about mental health research!")
        print("Type 'quit' to exit, 'help' for commands")
        print("-"*60)
        
        while True:
            try:
                question = input("\n❓ Your question: ").strip()
                
                if question.lower() in ['quit', 'exit', 'q']:
                    print("👋 Goodbye!")
                    break
                
                if question.lower() == 'help':
                    self._show_help()
                    continue
                
                if not question:
                    continue
                
                response = self.query(question)
                
                self._display_response(response)
                
            except KeyboardInterrupt:
                print("\n👋 Goodbye!")
                break
            except Exception as e:
                print(f"❌ Error: {e}")
    
    def _show_help(self):
        """Show help information"""
        print("\n📖 Commands:")
        print("  - Ask any question about mental health research")
        print("  - 'quit' or 'exit' to leave")
        print("  - 'help' to show this message")
        print("\n💡 Example questions:")
        print("  - What is the prevalence of depression in diabetes patients?")
        print("  - How does anxiety affect adolescents?")
        print("  - What are the risk factors for mental health issues?")
    
    def _display_response(self, response: RAGResponse):
        """Display formatted response"""
        print(f"\n🤖 **Answer** (Confidence: {response.confidence:.2f}):")
        print("-" * 50)
        print(response.answer)
        
        if response.sources:
            print(f"\n📚 **Sources** ({len(response.sources)} chunks):")
            print("-" * 50)
            
            for i, source in enumerate(response.sources, 1):
                print(f"\n{i}. **{source.document_id}** - {source.section}")
                print(f"   Score: {source.score:.3f}")
                print(f"   Content: {source.content[:200]}...")
                if source.paper_title != "Unknown Paper":
                    print(f"   Paper: {source.paper_title}")

def main():
    """Main execution function"""
    
    if not os.getenv("OPENROUTER_API_KEY"):
        print("\n Error!!; key not found in env")
        return
    
    rag_engine = RAGQueryEngine(
        vector_db_path="D:/PsyWiz/vector_db",
        collection_name="psywiz_papers",
        retrieval_k=6,
        llm_model="google/gemini-2.0-flash-exp:free"
    )
    
    test_queries = [
        "What is the association between household dietary diversity and depression symptoms in adolescents from rural Pakistan?", #specific research queries
        
    ]
    
    print("\n" + "="*60)
    print("🧪 TESTING RAG PIPELINE WITH OPENROUTER")
    print("="*60)
    
    for query in test_queries:
        print(f"\n🔍 Query: {query}")
        try:
            response = rag_engine.query(query)
            rag_engine._display_response(response)
        except Exception as e:
            print(f"❌ Error processing query: {e}")
        print("\n" + "-"*60)
    
    # rag_engine.interactive_query()

if __name__ == "__main__":
    main()

🔧 Initializing RAG Query Engine with OpenRouter...
📊 Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
🗄️  Connecting to vector database: D:/PsyWiz/vector_db
📚 Found 87 documents in vector database
🤖 Initializing OpenRouter LLM...
✅ Connected to OpenRouter with model: google/gemini-2.0-flash-exp:free
✅ RAG Query Engine ready!

🧪 TESTING RAG PIPELINE WITH OPENROUTER

🔍 Query: What is the association between household dietary diversity and depression symptoms in adolescents from rural Pakistan?
🔍 Processing query: What is the association between household dietary diversity and depression symptoms in adolescents from rural Pakistan?
📖 Retrieved 6 relevant chunks

🤖 **Answer** (Confidence: 0.63):
--------------------------------------------------
Error calling OpenRouter API: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '

---

## APPROACH 2: Native RAG Using LLAMA INDEX WRAPPER

In [ ]:
%pip install llama-index llama-index-llms-openrouter llama-index-vector-stores-chroma llama-index-embeddings-huggingface

%pip install llama-index-llms-gemini

In [ ]:
import os
import chromadb
from typing import Dict, List
from dataclasses import dataclass
from dotenv import load_dotenv

from llama_index.core import VectorStoreIndex, Settings, get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext
from llama_index.llms.openrouter import OpenRouter

from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.gemini import Gemini

load_dotenv()

@dataclass
class RAGResponse:
    answer: str
    sources: List[Dict]
    query: str
    confidence: float

class LlamaIndexRAGEngine:
    
    def __init__(self, 
                 vector_db_path: str = "D:/PsyWiz/vector_db",
                 collection_name: str = "psywiz_papers",
                 retrieval_k: int = 6,
                 llm_model: str = "google/gemini-2.0-flash-exp:free"):
        
        self.vector_db_path = vector_db_path
        self.collection_name = collection_name
        self.retrieval_k = retrieval_k
        
        print("🔧 Initializing LlamaIndex RAG Engine...")
        self._setup_components(llm_model)
        self._build_query_engine()
        print("✅ LlamaIndex RAG Engine ready!")
    
    # def _setup_components(self, llm_model: str):
        
    #     # 1. Setup LLM
    #     print(f"🤖 Initializing OpenRouter LLM: {llm_model}")
    #     Settings.llm = OpenRouter(
    #         model=llm_model,
    #         api_key=os.getenv("OPENROUTER_API_KEY"),
    #         api_base="https://openrouter.ai/api/v1",
    #         max_tokens=2500,
    #         temperature=0.3
    #     )
    def _setup_components(self, llm_model: str):
        
        # 1. Setup LLM - Changed to Gemini
        print(f"🤖 Initializing Google AI Studio LLM: {llm_model}")
        Settings.llm = Gemini(
            model=llm_model,
            api_key=os.getenv("GEMINI_API_KEY"),  # Changed from OPENROUTER_API_KEY
            max_tokens=2500,
            temperature=0.3
        )
        
        # 2. Setup Embeddings - Fixed to use HuggingFaceEmbedding
        print("📊 Loading embedding model...")
        Settings.embed_model = HuggingFaceEmbedding(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
        
        # 3. Connect to existing Chroma DB
        print(f"🗄️ Connecting to ChromaDB...")
        client = chromadb.PersistentClient(path=self.vector_db_path)
        collection = client.get_collection(name=self.collection_name)
        print(f"📚 Found {collection.count()} documents")
        
        # 4. Create vector store
        self.vector_store = ChromaVectorStore(chroma_collection=collection)
        self.storage_context = StorageContext.from_defaults(vector_store=self.vector_store)
        
        # 5. Create index from existing vector store
        self.index = VectorStoreIndex.from_vector_store(
            vector_store=self.vector_store,
            storage_context=self.storage_context
        )
    
    def _build_query_engine(self):
        """Build the query engine with custom settings"""
        
        retriever = VectorIndexRetriever(
            index=self.index,
            similarity_top_k=self.retrieval_k,
        )
        
        response_synthesizer = get_response_synthesizer(
            response_mode="compact",
            use_async=False,
        )
        
        # Create query engine
        self.query_engine = RetrieverQueryEngine(
            retriever=retriever,
            response_synthesizer=response_synthesizer,
            node_postprocessors=[
                SimilarityPostprocessor(similarity_cutoff=0.3)
            ],
        )
        
        # FIXED: Use proper LlamaIndex prompt template
        from llama_index.core.prompts import PromptTemplate
        
        qa_prompt_str = """You are an expert medical research assistant. Answer the question based ONLY on the provided research paper excerpts.

    INSTRUCTIONS:
    - Provide accurate, evidence-based answers
    - Cite specific findings from the papers
    - Include statistical data when available (prevalence rates, confidence intervals, etc.)
    - Acknowledge limitations or gaps in the provided evidence
    - If the context doesn't contain sufficient information, clearly state this
    - Use a professional, academic tone

    Context information is below:
    ---------------------
    {context_str}
    ---------------------

    Question: {query_str}

    Evidence-based Answer:"""
        
        qa_prompt = PromptTemplate(qa_prompt_str)
        
        # Update with proper prompt template object
        self.query_engine.update_prompts({
            "response_synthesizer:text_qa_template": qa_prompt
        })
    
    def query(self, question: str) -> RAGResponse:

        print(f"🔍 Processing query: {question}")
        
        # Execute query
        response = self.query_engine.query(question)
        
        sources = []
        if hasattr(response, 'source_nodes'):
            print(f"📖 Retrieved {len(response.source_nodes)} relevant chunks")
            
            total_context = ""
            for i, node in enumerate(response.source_nodes):
                total_context += node.text + "\n\n"
                sources.append({
                    'chunk_id': node.metadata.get('chunk_id', f'chunk_{i}'),
                    'content': node.text[:200] + "...",
                    'score': node.score if hasattr(node, 'score') else 0.0,
                    'document_id': node.metadata.get('document_id', 'unknown'),
                    'section': node.metadata.get('section', 'Unknown'),
                    'paper_title': node.metadata.get('paper_title', 'Unknown Paper')
                })
            
            # Token counting estimation
            def estimate_tokens(text: str) -> int:
                return len(text) // 4
            
            context_tokens = estimate_tokens(total_context)
            response_tokens = estimate_tokens(str(response))
            total_tokens = context_tokens + response_tokens
            
            print(f"📊 TOKEN ANALYSIS:")
            print(f"   • Context tokens: ~{context_tokens:,}")
            print(f"   • Response tokens: ~{response_tokens:,}")
            print(f"   • Total tokens: ~{total_tokens:,}")
        
        avg_score = sum(s['score'] for s in sources) / len(sources) if sources else 0.0
        confidence = min(avg_score * 1.2, 1.0)
        
        return RAGResponse(
            answer=str(response),
            sources=sources,
            query=question,
            confidence=confidence
        )
    
    def interactive_query(self):
        print("\n" + "="*60)
        print("🧠 PsyWiz LlamaIndex RAG Engine - Interactive Mode")
        print("="*60)
        print("Ask questions about mental health research!")
        print("Type 'quit' to exit")
        print("-"*60)
        
        while True:
            try:
                question = input("\n❓ Your question: ").strip()
                
                if question.lower() in ['quit', 'exit', 'q']:
                    print("👋 Goodbye!")
                    break
                
                if not question:
                    continue
                
                # Process query
                response = self.query(question)
                
                # Display results
                print(f"\n🤖 **Answer** (Confidence: {response.confidence:.2f}):")
                print("-" * 50)
                print(response.answer)
                
                if response.sources:
                    print(f"\n📚 **Sources** ({len(response.sources)} chunks):")
                    print("-" * 50)
                    
                    for i, source in enumerate(response.sources, 1):
                        print(f"\n{i}. **{source['document_id']}** - {source['section']}")
                        print(f"   Score: {source['score']:.3f}")
                        print(f"   Content: {source['content']}")
                
            except KeyboardInterrupt:
                print("\n👋 Goodbye!")
                break
            except Exception as e:
                print(f"❌ Error: {e}")

def main():
    """Main execution function"""
    
    # # Check for API key
    # if not os.getenv("OPENROUTER_API_KEY"):
    #     print("\n❌ Error: OPENROUTER_API_KEY not found in environment variables")
    #     print("Please create a .env file with your OpenRouter API key:")
    #     print("OPENROUTER_API_KEY=your_api_key_here")
    #     return
    
    # # Initialize RAG engine
    # rag_engine = LlamaIndexRAGEngine(
    #     vector_db_path="D:/PsyWiz/vector_db",
    #     collection_name="psywiz_papers",
    #     retrieval_k=5,
    #     llm_model="google/gemini-2.0-flash-exp:free"
    # )

    if not os.getenv("GEMINI_API_KEY"):
        print("\nError--key !detected")
        print("Please create a .env file with your Google AI Studio API key:")
        print("GEMINI_API_KEY=your_api_key_here")
        return
    
    rag_engine = LlamaIndexRAGEngine(
        vector_db_path="D:/PsyWiz/vector_db",
        collection_name="psywiz_papers",
        retrieval_k=5,
        llm_model="gemini-2.0-flash"  # change
    )
    
    test_query = "How does maternal mental health mediate the relationship between food insecurity and adolescent anxiety?"
    
    print("\n" + "="*60)
    print("🧪 TESTING LLAMAINDEX RAG PIPELINE")
    print("="*60)
    
    try:
        response = rag_engine.query(test_query)
        print(f"\n🤖 **Answer** (Confidence: {response.confidence:.2f}):")
        print("-" * 50)
        print(response.answer)
        
        if response.sources:
            print(f"\n📚 **Sources** ({len(response.sources)} chunks):")
            for i, source in enumerate(response.sources, 1):
                print(f"{i}. {source['document_id']} - Score: {source['score']:.3f}")
                
    except Exception as e:
        print(f"error: {e}")
    
    # Start interactive mode
    # rag_engine.interactive_query()

if __name__ == "__main__":
    main()

🔧 Initializing LlamaIndex RAG Engine...
🤖 Initializing Google AI Studio LLM: gemini-2.0-flash


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_36344\3521444265.py:71: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/)
  Settings.llm = Gemini(


📊 Loading embedding model...
🗄️ Connecting to ChromaDB...
📚 Found 87 documents
✅ LlamaIndex RAG Engine ready!

🧪 TESTING LLAMAINDEX RAG PIPELINE
🔍 Processing query: How does maternal mental health mediate the relationship between food insecurity and adolescent anxiety?
📖 Retrieved 5 relevant chunks
📊 TOKEN ANALYSIS:
   • Context tokens: ~1,643
   • Response tokens: ~177
   • Total tokens: ~1,820

🤖 **Answer** (Confidence: 0.68):
--------------------------------------------------
Maternal mental health mediates the relationship between food insecurity and adolescent anxiety. The study found that higher scores on the mother's mental well-being measure were significantly associated with a decrease in anxiety symptoms in both boys (β = -0.395, p < 0.001) and girls (β = -0.468, p < 0.001). Additionally, food insecurity was negatively associated with maternal mental well-being in both boys (β = -0.840, p < 0.001) and girls (β = -0.704, p < 0.001). This suggests that food insecurity impacts m